In [1]:
# Imports and environment setup
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain


# Load environment variables
load_dotenv()


C:\Users\DELL\AppData\Local\Temp\ipykernel_3320\1119023110.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


True

In [2]:
# Load and split documents
print("Loading PDFs from 'pdfs' directory...")
loader = PyPDFDirectoryLoader("pdfs/")
documents = loader.load()
print(f"Loaded {len(documents)} pages in total.")

# Split texts into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=150,
    separators=["\n\n", "\n", " ", ""]
)

chunks = text_splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks.")

Loading PDFs from 'pdfs' directory...
Loaded 50 pages in total.
Split into 56 chunks.


In [3]:
# Initialize Embeddings and FAISS VectorStore
print("Generating embeddings and creating FAISS index...")

# Using a lightweight, fast open-source embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Create the vector database
vector_db = FAISS.from_documents(chunks, embeddings)

# Optional: Save the index locally so you don't have to embed every time
vector_db.save_local("faiss_parallel_computing_index")
print("FAISS index created and saved successfully!")

Generating embeddings and creating FAISS index...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2167.47it/s]


FAISS index created and saved successfully!


In [4]:
# Action 1 - Retrieve only possible chunks
retriever = vector_db.as_retriever(search_kwargs={"k": 3}) # Retrieve top 3 chunks

query = "What is the difference between shared memory and distributed memory in parallel computing?"

print(f"Query: {query}\n")
print("--- Retrieving Possible Chunks ---\n")

retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs):
    print(f"Chunk {i+1} (Source: {doc.metadata.get('source', 'Unknown')}):")
    print("-" * 40)
    print(doc.page_content)
    print("-" * 40 + "\n")

Query: What is the difference between shared memory and distributed memory in parallel computing?

--- Retrieving Possible Chunks ---

Chunk 1 (Source: pdfs\lecture 2.pdf):
----------------------------------------
Non-Shared (Distributed)
 Memory
Advantages: 
(1) Memory is scalable with number of processors. Increase 
the number of processors and the size of memory increases 
proportionately. 
(2) Each processor can rapidly access its own memory 
without interference and without the overhead incurred with 
trying to maintain cache coherency. 
(3) Cost effectiveness: can use commodity, off-the-shelf 
processors and networking. 
Disadvantages: 
(1) The programmer is responsible for many of the details 
associated with data communication between processors. 
(2) It may be difficult to map existing data structures, based 
on global memory, to this memory organization.
----------------------------------------

Chunk 2 (Source: pdfs\the Lecture 2 notes.pdf):
---------------------------------

In [7]:
#  Action 2 Setup - LLM and Strict Prompt
# Initialize ChatGroq with the specified model
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model="openai/gpt-oss-120b",
    max_tokens=1024
)

# Strict instruction prompt to prevent hallucination
prompt_template = """
You are an expert Teaching Assistant in Parallel Computing. 
Use ONLY the provided context below to answer the user's question. 

Strict Rules:
1. If the answer is not contained within the context provided, you MUST say exactly: " I don't know, this information is not in the context".
2. Do not invent, guess, or hallucinate information. 
3. Keep the answer clear and to the point.

Context:
{context}

Question: {input}

Answer:
"""

prompt = PromptTemplate(
    input_variables=["context", "input"],
    template=prompt_template
)

In [8]:
#  Build and run the Full RAG chain
# 1. Create a chain that passes the retrieved documents to the LLM
document_chain = create_stuff_documents_chain(llm, prompt)

# 2. Combine with the retriever to form the complete RAG chain
rag_chain = create_retrieval_chain(retriever, document_chain)

print("--- Running Full RAG Pipeline ---\n")

# Test 1: A question likely in the slides
test_query_1 = "What is the difference between shared memory and distributed memory in parallel computing?"
print(f"Question: {test_query_1}")
response_1 = rag_chain.invoke({"input": test_query_1})
print(f"Answer: {response_1['answer']}\n")

# Test 2: A question out of context to test the guardrails
test_query_2 = "How to bake a chocolate cake?"
print(f"Question: {test_query_2}")
response_2 = rag_chain.invoke({"input": test_query_2})
print(f"Answer: {response_2['answer']}")

--- Running Full RAG Pipeline ---

Question: What is the difference between shared memory and distributed memory in parallel computing?
Answer: **Shared memory**  
- One global memory space that all processors can address directly.  
- Programming is easy because data are accessed with ordinary pointers and communication is just memory reads/writes.  
- Fast data sharing (direct memory access).  
- Drawbacks: limited scalability (traffic and cache‑coherency overhead grow as CPUs are added), need for explicit synchronization, and high hardware cost/complexity for large systems.  

**Distributed (non‑shared) memory**  
- Each processor has its own private local memory; processors exchange data only via a network.  
- Memory size scales linearly with the number of processors, and local memory accesses are fast and free of cache‑coherency issues.  
- Drawbacks: the programmer must explicitly manage communication between processors and map data structures to the distributed layout, which is